# Postprocess a bunch of data for Zenodo

In [101]:
import cosmic
from cosmic.output import load_initC, save_initC
import h5py as h5
import os
import pandas as pd
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from importlib import reload
from astropy.coordinates import SkyCoord
import seaborn as sns
from copy import copy

import legwork as lw
import gala.potential as gp
import cogsworth

import sys
sys.path.append("../helpers")

import detections
import const
import plotting
import uncertainties as unc

In [ ]:
for variation in const.VARIATIONS:
    print(f"Variation: {variation}")
    for dco_type in const.DCO_TYPES:
        print(f"  DCO type: {dco_type}")

        sub_variation = [("", "optimistic"), ("_pessimistic", variation)] if variation == "fiducial" else [("", variation)]

        for sub_var, label in sub_variation:
            in_band = cogsworth.pop.load(f"/mnt/ceph/users/twagg/lisa-dcos/{variation}/{dco_type}{sub_var}_in_band.h5")

            any_detectable = (
                (in_band.bpp["snr_lisa_8yr_initial_pos"] > 12)
                | (in_band.bpp["snr_lisa_8yr_final_pos"] > 12)
                | (in_band.bpp["snr_decigo_8yr_initial_pos"] > 12)
                | (in_band.bpp["snr_decigo_8yr_final_pos"] > 12)
            )

            present_day_stats = in_band.bpp.loc[any_detectable, ["mass_1", "mass_2", "sep_present_day", "ecc_present_day", "weights", "MW_instance", "snr_lisa_8yr_initial_pos", "snr_lisa_8yr_final_pos", "snr_decigo_8yr_initial_pos", "snr_decigo_8yr_final_pos"]]
            present_day_stats.rename(columns={"sep_present_day": "sep", "ecc_present_day": "ecc"}, inplace=True)
            present_day_stats["final_pos_x"] = in_band.final_pos[any_detectable, 0].value
            present_day_stats["final_pos_y"] = in_band.final_pos[any_detectable, 1].value
            present_day_stats["final_pos_z"] = in_band.final_pos[any_detectable, 2].value
            present_day_stats["initial_pos_x"] = in_band.initial_galaxy.x[any_detectable].value
            present_day_stats["initial_pos_y"] = in_band.initial_galaxy.y[any_detectable].value
            present_day_stats["initial_pos_z"] = in_band.initial_galaxy.z[any_detectable].value

            detectable_initC = in_band.initC[any_detectable]

            f_detect = pd.read_hdf(f"/mnt/ceph/users/twagg/lisa-dcos/{variation}/{dco_type}{sub_var}_f_detect.h5")
            f_detect = f_detect.loc[:, ["lisa_8yr_initial_pos_SNRgt7", "lisa_8yr_final_pos_SNRgt7", "lisa_8yr_initial_pos_SNRgt12", "lisa_8yr_final_pos_SNRgt12"]]
            with h5.File(f"/mnt/ceph/users/twagg/lisa-dcos/{variation}/{dco_type}{sub_var}_f_detect.h5", "r") as f:
                total_weights = f["total_weights"][...]

            group = f"{label}/{dco_type}"
            present_day_stats.to_hdf("/mnt/ceph/users/twagg/lisa-dcos/zenodo_data/upload.h5", key=f"{group}/present_day_stats", mode="a")
            save_initC(filename="/mnt/ceph/users/twagg/lisa-dcos/zenodo_data/upload.h5", initC=detectable_initC, key=f"{group}/initC", settings_key=f"{group}/initC_settings")

            f_detect.to_hdf("/mnt/ceph/users/twagg/lisa-dcos/zenodo_data/upload.h5", key=f"{group}/f_detect", mode="a")
            with h5.File(f"/mnt/ceph/users/twagg/lisa-dcos/zenodo_data/upload.h5", "a") as f:
                f.create_dataset(f"{group}/total_weights", data=total_weights)


pd.read_hdf("/mnt/ceph/users/twagg/lisa-dcos/normalisations.h5").to_hdf("/mnt/ceph/users/twagg/lisa-dcos/zenodo_data/upload.h5", key="n_targets", mode="a")

print("Done!")

cogsworth warning: file was saved with COSMIC v4.2.1 but you are using v4.2.2


Variation: fiducial
  DCO type: BHBH


cogsworth warning: file was saved with COSMIC v4.2.1 but you are using v4.2.2
cogsworth warning: file was saved with COSMIC v4.2.1 but you are using v4.2.2


  DCO type: BHNS


cogsworth warning: file was saved with COSMIC v4.2.1 but you are using v4.2.2
cogsworth warning: file was saved with COSMIC v4.2.1 but you are using v4.2.2


...

  DCO type: BHWD


cogsworth warning: file was saved with COSMIC v4.2.1 but you are using v4.2.2


  DCO type: NSNS


cogsworth warning: file was saved with COSMIC v4.2.1 but you are using v4.2.2


  DCO type: NSWD
Done!
